In [0]:
bronze_df = spark.table("bronze_nba_shot_logs")

display(bronze_df.limit(5))

In [0]:
silver_df = bronze_df

for old_col in bronze_df.columns:   
    new_col = (                             # standardize to snake_case
        old_col.strip()
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )
    silver_df = silver_df.withColumnRenamed(old_col, new_col)

display(silver_df.limit(5))

In [0]:
silver_df.columns

In [0]:
from pyspark.sql.functions import col

silver_df = (
    silver_df
    .withColumn("game_id", col("game_id").cast("int"))
    .withColumn("shot_number", col("shot_number").cast("int"))
    .withColumn("period", col("period").cast("int"))
    .withColumn("shot_clock", col("shot_clock").cast("double"))
    .withColumn("dribbles", col("dribbles").cast("double"))
    .withColumn("touch_time", col("touch_time").cast("double"))
    .withColumn("shot_dist", col("shot_dist").cast("double"))
    .withColumn("pts_type", col("pts_type").cast("int"))
    .withColumn("close_def_dist", col("close_def_dist").cast("double"))
    .withColumn("fgm", col("fgm").cast("int"))
    .withColumn("pts", col("pts").cast("int"))
    .withColumn("player_id", col("player_id").cast("int"))
)

In [0]:
silver_df = silver_df.withColumn("label", col("fgm").cast("int"))       # create label for ML

display(
    silver_df
    .select("shot_result", "fgm", "label")
    .limit(20)
)

In [0]:
from pyspark.sql.functions import col, when, hour, minute, second

silver_df = (
    silver_df
    .withColumn(
        "game_clock_seconds_remaining",
        (hour(col("game_clock")) * 60) + minute(col("game_clock"))
    )
    .withColumn(
        "total_seconds_left_in_game",
        ((4 - col("period")) * 720) + col("game_clock_seconds_remaining")
    )
    .withColumn(
        "is_shot_clock_missing",
        when(col("shot_clock").isNull(), 1).otherwise(0)
    )
    .withColumn(
        "shot_clock_adjusted",
        when(
            col("shot_clock").isNull(),
            col("game_clock_seconds_remaining").cast("double")
        ).otherwise(col("shot_clock"))
    )
)

In [0]:
display(
    silver_df
    .select(
        "period",
        "game_clock",
        "game_clock_seconds_remaining",
        "total_seconds_left_in_game",
        "shot_clock",
        "is_shot_clock_missing",
        "shot_clock_adjusted"
    )
    .limit(25)
)

display(
    silver_df
    .select(
        "period",
        "game_clock",
        "game_clock_seconds_remaining",
        "total_seconds_left_in_game",
        "shot_clock",
        "is_shot_clock_missing",
        "shot_clock_adjusted"
    )
    .filter(col("is_shot_clock_missing") == 1)
    .limit(25)
)

In [0]:
from pyspark.sql.functions import sum as spark_sum, when, col

required_cols = [
    "shot_dist",
    "shot_clock",
    "game_clock",
    "game_clock_seconds_remaining",
    "total_seconds_left_in_game",
    "shot_clock_adjusted",
    "is_shot_clock_missing",
    "dribbles",
    "touch_time",
    "close_def_dist",
    "period",
    "pts_type",
    "label"
]

missing_summary = silver_df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c + "_nulls")
    for c in required_cols
])

display(missing_summary)

In [0]:
silver_clean_df = silver_df.filter(     # clean out nulls and invalids
    (col("shot_dist").isNotNull()) &
    (col("game_clock").isNotNull()) &
    (col("game_clock_seconds_remaining").isNotNull()) &
    (col("total_seconds_left_in_game").isNotNull()) &
    (col("shot_clock_adjusted").isNotNull()) &
    (col("is_shot_clock_missing").isNotNull()) &
    (col("dribbles").isNotNull()) &
    (col("touch_time").isNotNull()) &
    (col("close_def_dist").isNotNull()) &
    (col("period").isNotNull()) &
    (col("pts_type").isNotNull()) &
    (col("label").isNotNull()) &
    (col("shot_dist") >= 0) &
    (col("game_clock_seconds_remaining") >= 0) &
    (col("game_clock_seconds_remaining") <= 720) &          # 12 * 60
    (col("total_seconds_left_in_game") >= 0) &
    (col("total_seconds_left_in_game") <= 2880) &           # 12 * 4 * 60
    (col("shot_clock_adjusted") >= 0) &
    (col("shot_clock_adjusted") <= 24) &
    (col("dribbles") >= 0) &
    (col("touch_time") >= 0) &
    (col("close_def_dist") >= 0) &
    (col("period").between(1, 4)) &
    (col("label").isin(0, 1)) &
    (col("is_shot_clock_missing").isin(0, 1))
)

display(silver_clean_df.limit(10))

In [0]:
bronze_count = bronze_df.count()
silver_count = silver_clean_df.count()

print(f"Bronze row count: {bronze_count}")
print(f"Silver clean row count: {silver_count}")
print(f"Rows removed during cleaning: {bronze_count - silver_count}")

In [0]:
silver_clean_df.select(
    "shot_dist",
    "shot_clock",
    "dribbles",
    "touch_time",
    "close_def_dist",
    "label"
).summary().show()

In [0]:
from pyspark.sql.functions import avg, count

make_rate_df = silver_clean_df.select(
    count("*").alias("total_shots"),
    avg("label").alias("make_rate"),
    avg("shot_dist").alias("avg_shot_distance"),
    avg("close_def_dist").alias("avg_defender_distance")
)

display(make_rate_df)

In [0]:
spark.sql("DROP TABLE IF EXISTS silver_nba_shot_logs_cleaned")
silver_clean_df.write.mode("overwrite").saveAsTable("silver_nba_shot_logs_cleaned")

In [0]:
spark.sql("""
SELECT 
  COUNT(*) AS row_count,
  AVG(label) AS make_rate,
  AVG(shot_dist) AS avg_shot_distance,
  AVG(close_def_dist) AS avg_defender_distance
FROM silver_nba_shot_logs_cleaned
""").show()